In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

YOLO_ROT_DIR = BASE_DIR / "yolo" / "rey_rotation"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("YOLO_ROT_DIR existe:", YOLO_ROT_DIR.exists())
print("Ruta dataset YOLO rotation:", YOLO_ROT_DIR)

In [ ]:
clases = ["rot_m90", "rot_0", "rot_90", "rot_180"]
splits = ["train", "val", "test"]

for split in splits:
    print("\n", split.upper())
    
    for clase in clases:
        carpeta = YOLO_ROT_DIR / split / clase
        n_png = len(list(carpeta.glob("*.png")))
        n_jpg = len(list(carpeta.glob("*.jpg")))
        total = n_png + n_jpg
        
        print(clase, ":", total)

In [ ]:
model = YOLO("yolo11n-cls.pt")

In [ ]:
results = model.train(
    data=str(YOLO_ROT_DIR),
    epochs=30,
    imgsz=160,
    batch=16,
    device="cpu",
    project=str(BASE_DIR / "outputs" / "yolo_runs"),
    name="rey_rotation_yolo11n_cls",
    exist_ok=True
)

In [ ]:
best_model_path = BASE_DIR / "outputs" / "yolo_runs" / "rey_rotation_yolo11n_cls" / "weights" / "best.pt"

print("Best model existe:", best_model_path.exists())
print(best_model_path)

In [ ]:
best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(YOLO_ROT_DIR),
    split="test",
    imgsz=160,
    device="cpu"
)

In [ ]:
test_dir = YOLO_ROT_DIR / "test"

imagenes_test = []
clases_reales = []

for clase in clases:
    carpeta = test_dir / clase
    
    archivos = list(carpeta.glob("*.png")) + list(carpeta.glob("*.jpg"))
    
    for archivo in archivos:
        imagenes_test.append(archivo)
        clases_reales.append(clase)

print("Total imágenes test:", len(imagenes_test))

for i in range(5):
    print(imagenes_test[i].name, "->", clases_reales[i])

In [ ]:
clases_predichas = []
confianzas = []

for img_path in imagenes_test:
    pred = best_model.predict(
        source=str(img_path),
        imgsz=160,
        device="cpu",
        verbose=False
    )
    
    r = pred[0]
    
    idx_pred = int(r.probs.top1)
    conf_pred = float(r.probs.top1conf)
    
    nombre_pred = r.names[idx_pred]
    
    clases_predichas.append(nombre_pred)
    confianzas.append(conf_pred)

print("Primeras predicciones:")
for i in range(10):
    print(clases_reales[i], "->", clases_predichas[i], "conf:", confianzas[i])

In [ ]:
acc_yolo = accuracy_score(clases_reales, clases_predichas)

print("Accuracy YOLO rotation:", acc_yolo)

cm_yolo = confusion_matrix(
    clases_reales,
    clases_predichas,
    labels=clases
)

print("\nMatriz de confusión:")
print(cm_yolo)

print("\nReporte de clasificación:")
print(classification_report(
    clases_reales,
    clases_predichas,
    labels=clases
))

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(cm_yolo)

plt.title("Matriz de confusión - Rotación YOLO")
plt.xlabel("Predicción")
plt.ylabel("Real")

plt.xticks(np.arange(len(clases)), clases, rotation=45)
plt.yticks(np.arange(len(clases)), clases)

for i in range(len(clases)):
    for j in range(len(clases)):
        plt.text(j, i, cm_yolo[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()

fig_path = FIGURES_DIR / "rey_rotation_yolo_confusion_matrix.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
df_resultados_yolo = pd.DataFrame({
    "imagen": [p.name for p in imagenes_test],
    "clase_real": clases_reales,
    "clase_pred": clases_predichas,
    "confianza": confianzas
})

csv_path = TABLES_DIR / "rey_rotation_yolo_resultados_test.csv"
df_resultados_yolo.to_csv(csv_path, index=False)

print("Resultados guardados en:", csv_path)
display(df_resultados_yolo.head())

In [ ]:
tabla_yolo_rotation = pd.DataFrame({
    "modelo": ["YOLO11n-cls"],
    "accuracy_test": [acc_yolo],
    "comentario": ["Clasificación de orientación con modelo YOLO preentrenado"]
})

tabla_path = TABLES_DIR / "rey_rotation_yolo_resumen_test.csv"
tabla_yolo_rotation.to_csv(tabla_path, index=False)

display(tabla_yolo_rotation)
print("Tabla guardada en:", tabla_path)

In [ ]:
acc_keras = 0.9248120188713074

tabla_comparativa_rotation = pd.DataFrame({
    "modelo": ["CNN Keras", "YOLO11n-cls"],
    "accuracy_test": [acc_keras, acc_yolo],
    "tipo": ["CNN propia", "Modelo preentrenado YOLO classification"]
})

tabla_path = TABLES_DIR / "rey_rotation_comparativa_keras_yolo.csv"
tabla_comparativa_rotation.to_csv(tabla_path, index=False)

display(tabla_comparativa_rotation)
print("Tabla guardada en:", tabla_path)